In [1]:
def check_guess(secret, guess):
    result = [None, None, None, None, None]
    secret_list = list(secret)

    for i in range(len(guess)):
        if guess[i] == secret_list[i]:
            result[i] = "green"
            secret_list[i] = None

    for i in range(len(guess)):
        if result[i] is None and guess[i] in secret_list:
            result[i] = "yellow"
            secret_list[secret_list.index(guess[i])] = None
        elif result[i] is None:
            result[i] = "gray"

    return result

print(check_guess("apple", "arppp"))

['green', 'gray', 'green', 'yellow', 'gray']


In [ ]:
import tkinter as tk
import random

BG        = "#121213"
TILE_BG   = "#121213"
TILE_FG   = "#ffffff"
BORDER    = "#565758"
COLOR_MAP = {"green": "#538d4e", "yellow": "#b59f3b", "gray": "#3a3a3c"}

with open("wordlist.txt") as f:
    valid_words = set(w.strip().lower() for w in f)
with open("wordlist_answers.txt") as f:
    answer_words = [w.strip() for w in f]

root = tk.Tk()
root.title("Dordle")
root.configure(bg=BG)

container = tk.Frame(root, bg=BG)
container.pack(expand=True, padx=20, pady=20)

title_label = tk.Label(container, text="DORDLE",
                       font=("Helvetica", 24, "bold"),
                       bg=BG, fg="#ffffff")
title_label.grid(row=0, column=0, columnspan=11, pady=(0, 8))

separator = tk.Frame(container, bg="#565758", height=1)
separator.grid(row=1, column=0, columnspan=11, sticky="ew", pady=(0, 12))

# Two grids side by side (7 rows x 5 cols, gap at col 5)
tiles = [[], []]
for board in range(2):
    col_offset = 6 * board
    for row in range(7):
        row_tiles = []
        for col in range(5):
            label = tk.Label(container, width=3, height=1,
                             font=("Helvetica", 14, "bold"),
                             bg=TILE_BG, fg=TILE_FG,
                             highlightbackground=BORDER,
                             highlightthickness=2)
            label.grid(row=row+2, column=col+col_offset, padx=2, pady=2)
            row_tiles.append(label)
        tiles[board].append(row_tiles)


vert_sep = tk.Frame(container, bg="#565758", width=1)
vert_sep.grid(row=2, column=5, rowspan=7, sticky="ns", padx=6)


msg_label = tk.Label(container, text="", font=("Helvetica", 12),
                     bg=BG, fg="#ffffff")
msg_label.grid(row=9, column=0, columnspan=11, pady=(8, 0))


restart_btn = tk.Button(container, text="Play Again",
                        font=("Helvetica", 12, "bold"),
                        bg="#538d4e", fg="white",
                        relief="flat", padx=10, pady=5)
restart_btn.grid(row=10, column=0, columnspan=11, pady=(6, 0))
restart_btn.grid_remove()


current_guess = []
current_row = 0
clear_msg_id = None
solved = [False, False]
secrets = [random.choice(answer_words), random.choice(answer_words)]
while secrets[0] == secrets[1]:
    secrets[1] = random.choice(answer_words)

def restart():
    global current_guess, current_row, solved, secrets
    current_guess = []
    current_row = 0
    solved = [False, False]
    secrets = [random.choice(answer_words), random.choice(answer_words)]
    while secrets[0] == secrets[1]:
        secrets[1] = random.choice(answer_words)
    for board in range(2):
        for row in tiles[board]:
            for tile in row:
                tile.config(text="", bg=TILE_BG, fg=TILE_FG,
                            highlightbackground=BORDER)
    msg_label.config(text="")
    restart_btn.grid_remove()
    root.bind("<Key>", on_key_press)

restart_btn.config(command=restart)

def end_game(message):
    global clear_msg_id
    if clear_msg_id:
        root.after_cancel(clear_msg_id)
        clear_msg_id = None
    msg_label.config(text=message)
    restart_btn.grid()

def on_key_press(event):
    global current_guess, current_row, clear_msg_id

    if event.keysym == "BackSpace":
        if current_guess:
            current_guess.pop()
            for board in range(2):
                if not solved[board]:
                    tiles[board][current_row][len(current_guess)].config(text="")

    elif event.keysym == "Return":
        if restart_btn.winfo_ismapped():
            restart()
            return
        if len(current_guess) == 5:
            guess_str = "".join(current_guess).lower()
            if guess_str not in valid_words:
                msg_label.config(text="Not a word")
                clear_msg_id = root.after(1500, lambda: msg_label.config(text=""))
                return
            msg_label.config(text="")

            for board in range(2):
                if not solved[board]:
                    result = check_guess(secrets[board], guess_str)
                    for i, outcome in enumerate(result):
                        tiles[board][current_row][i].config(bg=COLOR_MAP[outcome], fg="white")
                    if all(r == "green" for r in result):
                        solved[board] = True

            current_guess = []
            current_row += 1

            if all(solved):
                end_game("You solved both!")
            elif current_row > 6:
                unrevealed = [s.upper() for s, ok in zip(secrets, solved) if not ok]
                end_game("Words: " + " & ".join(unrevealed))

    elif event.char.isalpha() and len(current_guess) < 5:
        letter = event.char.upper()
        for board in range(2):
            if not solved[board]:
                tiles[board][current_row][len(current_guess)].config(text=letter)
        current_guess.append(letter)

root.bind("<Key>", on_key_press)

root.mainloop()
